# 05g - HayFlow-Hines regularized optimization audit

This diagnostic distinguishes an optimizer/conditioning failure from an inadequate frozen representation after 05f. It uses a closed-form float64 dual ridge path, compares ranks 64 and 96, bounds residual voltage corrections, and keeps held-out boundary-voltage targets sealed until a candidate passes train, development, coefficient, clipping, and feature-scale gates. It never launches full training.

## 1. Coherent checkout and GPU runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml', 'matplotlib'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire 05g.'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0), 'experiment': 'regularized train-first optimization audit'})

## 2. Inputs
Servono il composite targeted, il dataset base e gli artefatti esatti 05b, 05c, 05d, 05e e 05f. ZIP originali e directory Kaggle estratte sono accettati; i membri critici vengono verificati crittograficamente.

In [ ]:
import shutil, zipfile
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination)
    marker = destination / '.source_size'
    stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp:
        return destination
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp)
    return destination

def first_existing(candidates, message):
    found = next((Path(p).resolve() for p in candidates if Path(p).exists()), None)
    assert found is not None, message
    return found

topup_candidates = [Path(os.environ['HAYFLOW_TOPUP_V3']).expanduser()] if os.environ.get('HAYFLOW_TOPUP_V3') else []
topup_candidates += list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates += [p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE = first_existing(topup_candidates, 'Top-up BAP v3 non trovato.')
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05g_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'))
assert len(manifest_candidates) == 1, manifest_candidates
COMPOSITE_MANIFEST = manifest_candidates[0]

base_candidates = [Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else []
base_candidates += [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]
base_candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE = first_existing(base_candidates, 'Dataset base targeted v1.1 non trovato.')

def artifact_source(env_name, zip_name, directory_markers, message):
    candidates = [Path(os.environ[env_name]).expanduser()] if os.environ.get(env_name) else []
    candidates += list(INPUT_ROOT.rglob(zip_name))
    for marker in directory_markers:
        candidates += [p.parent for p in INPUT_ROOT.rglob(marker)]
    return first_existing(candidates, message)

CHECKPOINT_05B_SOURCE = artifact_source('HAYFLOW_05B_ARTIFACT', 'hayflow_hines_canary_v2.zip', ['canary_models.pt'], 'Artefatto 05b non trovato.')
if CHECKPOINT_05B_SOURCE.name == 'checkpoints': CHECKPOINT_05B_SOURCE = CHECKPOINT_05B_SOURCE.parent
ARTIFACT_05C_SOURCE = artifact_source('HAYFLOW_05C_ARTIFACT', 'hayflow_hines_causal_isolation.zip', ['checkpoint_forensics.json'], 'Artefatto 05c non trovato.')
ARTIFACT_05D_SOURCE = artifact_source('HAYFLOW_05D_ARTIFACT', 'hayflow_hines_residual_conditioning.zip', ['free_residual_report.json'], 'Artefatto 05d non trovato.')
ARTIFACT_05E_SOURCE = artifact_source('HAYFLOW_05E_ARTIFACT', 'hayflow_hines_segment_capacity.zip', ['capacity_probe_report.json'], 'Artefatto 05e non trovato.')
ARTIFACT_05F_SOURCE = artifact_source('HAYFLOW_05F_ARTIFACT', 'hayflow_hines_segment_micro_canary.zip', ['micro_canary_report.json'], 'Artefatto 05f non trovato.')
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), '05b': str(CHECKPOINT_05B_SOURCE), '05c': str(ARTIFACT_05C_SOURCE), '05d': str(ARTIFACT_05D_SOURCE), '05e': str(ARTIFACT_05E_SOURCE), '05f': str(ARTIFACT_05F_SOURCE)})

## 3. Composite and cryptographic provenance preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now)
    percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9)
        eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05g][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True)
        hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
bundle_summary = {'valid': bool(bundle.manifest['valid']), 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count, 'physical_merge_performed': bool(bundle.manifest['physical_merge_performed'])}
display(bundle_summary)
assert bundle_summary['valid'] and bundle_summary['transition_count'] == 29880
assert not bundle_summary['physical_merge_performed']

In [ ]:
from src.hayflow_model import HinesCapacityConfig, HinesConditioningConfig, HinesIsolationConfig, HinesOptimizationAuditConfig, HinesPrototypeExperimentConfig, HinesSegmentCanaryConfig, HinesSegmentOptimizationAudit
raw = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_optimization_audit.yml').read_text())
model_config = HinesPrototypeExperimentConfig.from_mapping(raw['model_experiment'])
isolation_config = HinesIsolationConfig.from_mapping(raw['isolation'])
conditioning_config = HinesConditioningConfig.from_mapping(raw['conditioning'])
capacity_config = HinesCapacityConfig.from_mapping(raw['capacity'])
canary_config = HinesSegmentCanaryConfig.from_mapping(raw['micro_canary'])
audit_config = HinesOptimizationAuditConfig.from_mapping(raw['optimization_audit'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_optimization_audit')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
session = HinesSegmentOptimizationAudit(bundle, OUTPUT_DIR, model_config, isolation_config, conditioning_config, capacity_config, canary_config, audit_config, CHECKPOINT_05B_SOURCE, ARTIFACT_05C_SOURCE, ARTIFACT_05D_SOURCE, ARTIFACT_05E_SOURCE, ARTIFACT_05F_SOURCE, code_revision=REVISION)
prepare_report = session.prepare_optimization_audit()
display({'revision': REVISION, 'dataset': bundle.fingerprint, '05f': prepare_report['artifact_05f'], 'full_training_authorized': prepare_report['full_training_authorized']})
assert prepare_report['training_contract_blockers'] == []
assert not prepare_report['full_training_authorized']

## 4. Train support and sealed feature-scale audit
The development pair is excluded from train. The selector searches farther through train trajectories and reports whether genuine protocol-family diversity is available. Held-out boundary-voltage targets are not loaded here.

In [ ]:
support_report = session.build_optimization_support()
display({k: support_report[k] for k in ['valid', 'candidate_pair_count', 'available_protocol_family_count', 'protocol_diversity_available', 'selected_pair_count', 'selected_protocol_family_count', 'protocol_diversity_selected', 'support_sha256']})
assert support_report['valid']
scale_report = session.prepare_audit_data()
display(scale_report)
assert not scale_report['heldout_targets_materialized']

## 5. Oracle controls and regularized float64 train-first path
The direct residual oracle validates target/metric plumbing across all train pairs; the segment-bias control measures static memorization. The actual audit solves small dual ridge systems, compares rank 64/96 after SVD truncation, and rejects non-finite, oversized or clipped solutions. Progress and ETA are printed.

In [ ]:
oracle_report = session.run_oracle_controls()
display(oracle_report)
assert oracle_report['valid']
train_report = session.run_regularized_train_audit(oracle_report, scale_report)
display({'candidate_count': train_report['candidate_count'], 'safe_candidate_count': train_report['safe_candidate_count'], 'best_train_candidate': train_report['best_train_candidate'], 'selected_safe_candidate': train_report['selected_safe_candidate'], 'heldout_reveal_authorized': train_report['heldout_reveal_authorized']})

## 6. Conditional held-out reveal and final decision
Held-out boundary-voltage targets are loaded only if one candidate already passed every train, development and numerical-safety gate. Regardless of outcome, 05g remains diagnostic and cannot authorize full training.

In [ ]:
heldout_report = session.reveal_heldout_if_safe(train_report)
final_report = session.finalize_optimization_audit(scale_report, oracle_report, train_report, heldout_report)
display({'valid': final_report['valid'], 'decision': final_report['decision'], 'diagnosis': final_report['diagnosis'], 'heldout_gate': heldout_report, 'next_step': final_report['next_step']})
assert final_report['valid']
assert not final_report['full_training_authorized']

## 7. Inspect diagnostics

In [ ]:
display(pd.read_parquet(OUTPUT_DIR / 'regularized_audit_metrics.parquet').sort_values(['candidate_safe_for_heldout_reveal', 'development_voltage_rmse_mv', 'train_voltage_rmse_mv'], ascending=[False, True, True]))
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / 'optimization_audit.png')))

## 8. Create and download the audit ZIP

In [ ]:
from pathlib import Path
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_optimization_audit')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
payload = base64.b64encode(zip_path.read_bytes()).decode('ascii')
filename = zip_path.name
display(Javascript(f"""
const binary = atob('{payload}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print({'zip': str(zip_path), 'size_mib': round(zip_path.stat().st_size / 2**20, 2), 'download': 'avviato dal browser'})